In [1]:
!pip install black==24.3.0 --no-deps
!pip install pylint==2.17.7 --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 127.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.2/537.2 kB 53.7 MB/s  0:00:00


# Configuracion

In [2]:
team = 'RETAIL' # TODO 1: Colocar nombre del equipo: RETAIL, RIESGOS
name_ds = 'Hernandez Santiago' # TODO 2: Colocar mis apellidos y nombres
path_script = './inference.py'

In [3]:
import os
import boto3
from sagemaker import get_execution_role

account = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'
role_arn = get_execution_role()

os.environ['I_TEAM_RETAIL'], os.environ['I_CC_RETAIL'] = 'DS RETAIL', '9946100000'
os.environ['I_TEAM_RIESGOS'], os.environ['I_CC_RIESGOS'] = 'DS RIESGOS', '9810200000'

tags = [
    {'Key': 'I_RESPONSABLE_LT', 'Value': name_ds},
    {'Key': 'I_APLICACION', 'Value': 'SDLF'},
    {'Key': 'I_PROYECTO', 'Value': 'SDLF'},
    {'Key': 'I_AMBIENTE', 'Value': 'DEV'},
    {'Key': 'I_CUENTA', 'Value': account},
    {'Key': 'I_SIGLA', 'Value': 'SAN'},
    {'Key': 'I_TEAM', 'Value': os.environ[f'I_TEAM_{team}']},
    {'Key': 'I_CC', 'Value': os.environ[f'I_CC_{team}']},
]

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


# Formatear Script

In [4]:
!black --line-length 100 $path_script
!pylint --disable C0103 $path_script

Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/bin/black", line 3, in <module>
    from black import patched_main
  File "src/black/__init__.py", line 33, in <module>
ModuleNotFoundError: No module named 'mypy_extensions'
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/bin/pylint", line 6, in <module>
    sys.exit(run_pylint())
             ^^^^^^^^^^^^
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/__init__.py", line 33, in run_pylint
    from pylint.lint import Run as PylintRun
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/lint/__init__.py", line 19, in <module>
    from pylint.config.exceptions import ArgumentPreprocessingError
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/config/__init__.py", line 25, in <module>
    from pylint.config.arguments_provider import UnsupportedAction
  File "/home/ec2-user/anaconda3

# Crear Processor

| Instancia      | CPU Cores | Memoria (GB) |
|----------------|-----------|--------------|
| ml.m5.large    | 2         | 8            |
| ml.m5.xlarge   | 4         | 16           |
| ml.m5.2xlarge  | 8         | 32           |
| ml.m5.4xlarge  | 16        | 64           |
| ml.m5.12xlarge | 48        | 192          |
| ml.m5.24xlarge | 96        | 384          |

In [5]:
from sagemaker.processing import ScriptProcessor

processor = ScriptProcessor(instance_type='ml.m5.12xlarge',
                            volume_size_in_gb=30,
                            instance_count=1,
                            command=['python3'],
                            image_uri=image_uri,
                            role=role_arn,
                            tags=tags)

# Definir Entradas

In [6]:
from sagemaker.processing import ProcessingInput
inputs = []

inputs.append(ProcessingInput(
    input_name='code/utils',
    source='s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/utils/', # TODO 3: Colocar ubicacion de S3 del script 'utils.py'
    destination='/opt/ml/processing/input/code/utils',
))

inputs.append(ProcessingInput(
    input_name='data',
    source='s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/DATA_INFERENCIA_PILOTO/INFERENCIA/periodo=202604/', # TODO 4: Colocar ubicacion de S3 de los datos del periodo de inferencia
    destination='/opt/ml/processing/input/data',
))

inputs.append(ProcessingInput(
    input_name='models',
    source='s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/MODEL/calibrado_v1/', # TODO 5: Colocar ubicacion de S3 de los modelos entrenados
    destination='/opt/ml/processing/input/models',
))

# Definir Salidas

In [7]:
from sagemaker.processing import ProcessingOutput

outputs = []

outputs.append(ProcessingOutput(
    output_name='results',
    source='/opt/ml/processing/output/results',
    destination='s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/REPLICA_OUTPUT',
   # 's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/nsantilli/digitalizacion/REPLICA_OUTPUT/', # TODO 6: Colocar ubicacion de S3 donde van a estar los resultados
))

# Ejecutar Job

In [8]:
model = 'PLAFPJMINORISTA' # TODO 7: Colocar nombre de la rama del modelo
partition = '202604' # TODO 8: Colocar periodo de los datos

arguments = [
    '--model', model,
    '--table-score', f'scr_{model}',
    '--partition', partition,
]

processor.run(code=path_script,
              inputs=inputs,
              outputs=outputs,
              arguments=arguments)

INFO:sagemaker:Creating processing-job with name sagemaker-python3-2026-06-02-20-40-01-548


.........../usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.Float64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.UInt64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/io/parquet/arrow.py:144: FutureWarning: 'Parque

### Bibliotecas Instaladas

- `image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'`  

    ```
    catboost           1.1
    certifi            2022.9.24
    charset-normalizer 2.1.1
    cycler             0.11.0
    dask               2.11.0
    docopt             0.6.2
    fonttools          4.38.0
    fsspec             2022.11.0
    graphviz           0.20.1
    idna               3.4
    joblib             1.2.0
    kiwisolver         1.4.4
    lightgbm           3.3.3
    locket             1.0.0
    matplotlib         3.5.0
    numpy              1.23.5
    packaging          21.3
    pandas             1.5.1
    partd              1.3.0
    Pillow             9.3.0
    pip                22.0.4
    pipreqs            0.4.11
    plotly             5.11.0
    pyarrow            10.0.0
    pyparsing          3.0.9
    python-dateutil    2.8.2
    pytz               2022.6
    requests           2.28.1
    scikit-learn       1.1.2
    scipy              1.9.3
    seaborn            0.11.2
    setuptools         57.5.0
    setuptools-scm     7.0.5
    six                1.16.0
    tabulate           0.8.9
    tenacity           8.1.0
    threadpoolctl      3.1.0
    tomli              2.0.1
    toolz              0.12.0
    tqdm               4.62.3
    typing_extensions  4.4.0
    urllib3            1.26.12
    wheel              0.38.4
    xgboost            1.6.2
    yarg               0.1.9
    ```